# Kaki — Model A: Scam Message Classifier
Interactive walkthrough of `train.py` (the single training pipeline). Uses the same code, data and
evaluation protocol, so numbers here match `kaki_model.json`.

- **Data:** UCI SMS Spam Collection (Kaggle) + `scam_sg.csv` (Singapore-style, 4 languages)
- **Model:** binary word / Han-character n-gram logistic regression (`kaki_ml.features`)
- **Evaluation:** out-of-fold predictions; SG per-language figures rest on ~11–19 rows each and are low-confidence

In [ ]:
import os, sys, platform
ROOT = os.path.abspath('..')
sys.path.insert(0, ROOT)
os.chdir(ROOT)
print('Machine:', platform.machine(), '| Python', platform.python_version())
# pip install -r requirements-train.txt   (run once in a terminal)
import kaki_ml, train

In [ ]:
import pandas as pd
uci, uci_src = train.load_sms_spam()
msgs = pd.concat([uci, train.load_scam_sg()], ignore_index=True)
print('UCI source:', uci_src)
msgs.groupby(['source', 'lang', 'label']).size()

In [ ]:
pipe, metrics, scored = train.train_scam_model(msgs)
train.print_scam_metrics(metrics)

In [ ]:
# What the model and rules say about a few messages
for t in ['SPF Officer: arrest warrant. Transfer $50,000 to safe account 123-456789-0. CONFIDENTIAL.',
          'Your HealthHub appointment on 20 Oct at 10:30 AM at Toa Payoh Polyclinic is confirmed.',
          'Hi aunty! I need $2,000 urgently. New number, dont call my old phone.']:
    p = float(pipe.predict_proba([t])[0, 1]); hits = kaki_ml.rule_hits(t)
    print(f'{kaki_ml.decide(p, hits):10s} p={p:.2f} rules={hits}  {t[:70]}')

To produce the deployable artifacts (`models/`, `kaki_model.json`) run `python train.py` — it repeats these steps plus Model B and writes the SHA-256 manifest the API checks.